# 🤖 Phase 3: Deep Learning for EEG Classification

Three architectures implemented from scratch:

| Model | Architecture | Why |
|---|---|---|
| **EEGNet** | Depthwise separable CNN | Compact, generalizes across BCI tasks |
| **CNN-LSTM** | Spatial CNN + temporal LSTM | Captures local patterns + long-range dynamics |
| **EEG Conformer** | CNN + Transformer encoder | State-of-the-art: local + global dependencies |

All models operate on **raw epoch tensors** (channels × time), not hand-crafted features.

In [ ]:
import os
import json
import mne
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

mne.set_log_level('WARNING')

OUTPUT_PATH = Path(os.getcwd()) / 'processed'
MODEL_PATH  = Path(os.getcwd()) / 'models'
MODEL_PATH.mkdir(exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'✅ Using device: {DEVICE}')

tasks  = ['med1breath', 'med2', 'think1', 'think2']
SFREQ  = 128
T_LEN  = int(2.0 * SFREQ)   # 256 time points per epoch
N_CLS  = len(tasks)

with open(OUTPUT_PATH / 'subject_split.json') as f:
    split_info = json.load(f)

train_subjects = split_info['train']
test_subjects  = split_info['test']
print(f'   Train: {len(train_subjects)} subjects | Test: {len(test_subjects)} subjects')

## 📦 Dataset Loader

In [ ]:
class EEGEpochDataset(Dataset):
    """
    PyTorch Dataset that loads preprocessed MNE epochs.
    Returns tensors of shape (1, n_channels, T_LEN).
    """
    def __init__(self, subject_list, tasks, output_path, t_len=256):
        self.samples  = []   # list of (epoch_array, label_int)
        self.t_len    = t_len
        self.le       = LabelEncoder().fit(tasks)

        for subj in subject_list:
            for task in tasks:
                ep_file = output_path / f'{subj}_task-{task}_epochs.fif'
                if not ep_file.exists():
                    continue
                epochs = mne.read_epochs(str(ep_file), preload=True, verbose=False)
                data   = epochs.get_data()   # (n_epochs, n_ch, n_times)
                label  = self.le.transform([task])[0]

                for ep in data:
                    # Pad or truncate to fixed length
                    if ep.shape[1] >= t_len:
                        ep = ep[:, :t_len]
                    else:
                        ep = np.pad(ep, ((0,0), (0, t_len - ep.shape[1])))
                    self.samples.append((ep.astype(np.float32), label))

        self.n_channels = self.samples[0][0].shape[0] if self.samples else 64
        print(f'   Loaded {len(self.samples)} epochs | {self.n_channels} channels')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ep, label = self.samples[idx]
        x = torch.tensor(ep).unsqueeze(0)   # (1, n_ch, T)
        return x, label


print('⏳ Building datasets...')
train_ds  = EEGEpochDataset(train_subjects, tasks, OUTPUT_PATH, T_LEN)
test_ds   = EEGEpochDataset(test_subjects,  tasks, OUTPUT_PATH, T_LEN)
N_CH = train_ds.n_channels

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=0)
print(f'\n✅ DataLoaders ready | n_channels={N_CH} | T={T_LEN}')

## 🏗️ Model 1: EEGNet

Original paper: *Lawhern et al. (2018) — EEGNet: A Compact Convolutional Neural Network for EEG-based BCIs*

Key design:
- **Temporal conv** → captures EEG oscillations
- **Depthwise spatial conv** → learns spatial filters per channel
- **Separable conv** → efficient temporal summarization
- Very few parameters (~2K) → generalizes well with small data

In [ ]:
class EEGNet(nn.Module):
    """
    EEGNet (Lawhern et al. 2018).
    Input: (batch, 1, n_channels, T)
    Output: (batch, n_classes)
    """
    def __init__(self, n_classes=4, n_channels=64, T=256,
                 F1=8, D=2, F2=16, kern_len=64, dropout=0.5):
        super().__init__()

        # Block 1: Temporal convolution
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kern_len), padding=(0, kern_len // 2), bias=False),
            nn.BatchNorm2d(F1),
            # Depthwise spatial convolution (one filter per channel per F1 feature map)
            nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )

        # Block 2: Separable convolution
        self.block2 = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8),
                      groups=F1 * D, bias=False),   # depthwise
            nn.Conv2d(F1 * D, F2, (1, 1), bias=False),  # pointwise
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout),
        )

        # Flatten + classifier
        # Compute flattened size
        dummy = torch.zeros(1, 1, n_channels, T)
        with torch.no_grad():
            x = self.block1(dummy)
            x = self.block2(x)
        flat_size = x.view(1, -1).shape[1]

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, n_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return self.classifier(x)


eegnet = EEGNet(n_classes=N_CLS, n_channels=N_CH, T=T_LEN).to(DEVICE)
total_params = sum(p.numel() for p in eegnet.parameters())
print(f'✅ EEGNet | Parameters: {total_params:,}')
print(eegnet)

## 🏗️ Model 2: CNN-LSTM

CNN extracts **local spatial-temporal patterns**, LSTM captures **long-range temporal dynamics** — meditation state builds up over seconds, not milliseconds.

In [ ]:
class CNNLSTM(nn.Module):
    """
    Spatial CNN → temporal LSTM classifier.
    Input: (batch, 1, n_channels, T)
    Output: (batch, n_classes)
    """
    def __init__(self, n_classes=4, n_channels=64, T=256,
                 cnn_filters=32, lstm_hidden=64, dropout=0.4):
        super().__init__()

        # Spatial CNN: compress channels
        self.spatial_cnn = nn.Sequential(
            nn.Conv2d(1, cnn_filters, kernel_size=(n_channels, 1), bias=False),
            nn.BatchNorm2d(cnn_filters),
            nn.ELU(),
            nn.Dropout(dropout),
        )  # output: (batch, cnn_filters, 1, T)

        # Temporal CNN: capture short-range patterns
        self.temporal_cnn = nn.Sequential(
            nn.Conv1d(cnn_filters, cnn_filters, kernel_size=16, padding=8, bias=False),
            nn.BatchNorm1d(cnn_filters),
            nn.ELU(),
            nn.MaxPool1d(4),
            nn.Dropout(dropout),
        )  # output: (batch, cnn_filters, T//4)

        # LSTM: model temporal sequence
        self.lstm = nn.LSTM(
            input_size=cnn_filters,
            hidden_size=lstm_hidden,
            num_layers=2,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 64),  # *2 for bidirectional
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        # x: (batch, 1, n_ch, T)
        x = self.spatial_cnn(x)          # (batch, F, 1, T)
        x = x.squeeze(2)                  # (batch, F, T)
        x = self.temporal_cnn(x)          # (batch, F, T//4)
        x = x.permute(0, 2, 1)            # (batch, T//4, F) for LSTM
        out, (h, _) = self.lstm(x)
        # Use last hidden state from both directions
        h_last = torch.cat([h[-2], h[-1]], dim=1)  # (batch, 2*hidden)
        return self.classifier(h_last)


cnn_lstm = CNNLSTM(n_classes=N_CLS, n_channels=N_CH, T=T_LEN).to(DEVICE)
total_params = sum(p.numel() for p in cnn_lstm.parameters())
print(f'✅ CNN-LSTM | Parameters: {total_params:,}')

## 🏗️ Model 3: EEG Conformer

Combines **local CNN** (captures EEG oscillations) with **Transformer encoder** (captures global temporal context across the 2-second window). Based on Song et al. (2022).

In [ ]:
class PatchEmbedding(nn.Module):
    """Patch-based embedding: CNN tokenizer for EEG sequences."""
    def __init__(self, n_channels, emb_dim=40, patch_size=16, dropout=0.3):
        super().__init__()
        self.conv = nn.Sequential(
            # Temporal conv
            nn.Conv2d(1, 40, (1, 25), padding=(0, 12), bias=False),
            nn.BatchNorm2d(40),
            # Spatial depthwise conv
            nn.Conv2d(40, 40, (n_channels, 1), groups=40, bias=False),
            nn.BatchNorm2d(40),
            nn.ELU(),
            nn.AvgPool2d((1, patch_size)),
            nn.Dropout(dropout),
        )
        self.proj = nn.Linear(40, emb_dim)

    def forward(self, x):
        # x: (B, 1, n_ch, T)
        x = self.conv(x)          # (B, 40, 1, T//patch)
        x = x.squeeze(2)          # (B, 40, T//patch)
        x = x.permute(0, 2, 1)    # (B, T//patch, 40)
        return self.proj(x)        # (B, T//patch, emb_dim)


class EEGConformer(nn.Module):
    """
    EEG Conformer: CNN patch embedding + Transformer encoder.
    Input: (batch, 1, n_channels, T)
    Output: (batch, n_classes)
    """
    def __init__(self, n_classes=4, n_channels=64, T=256,
                 emb_dim=40, n_heads=4, n_layers=4,
                 patch_size=16, dropout=0.3):
        super().__init__()

        self.patch_embed = PatchEmbedding(n_channels, emb_dim, patch_size, dropout)
        n_patches = T // patch_size

        # CLS token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, emb_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches + 1, emb_dim) * 0.02)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim, nhead=n_heads,
            dim_feedforward=emb_dim * 4,
            dropout=dropout, batch_first=True,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(emb_dim)

        self.classifier = nn.Sequential(
            nn.Linear(emb_dim, emb_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(emb_dim // 2, n_classes)
        )

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)                               # (B, n_patches, emb)
        cls = self.cls_token.expand(B, -1, -1)               # (B, 1, emb)
        x   = torch.cat([cls, x], dim=1)                     # (B, n_patches+1, emb)
        x   = x + self.pos_embed[:, :x.shape[1]]
        x   = self.transformer(x)                            # (B, n_patches+1, emb)
        x   = self.norm(x[:, 0])                             # CLS token output
        return self.classifier(x)


conformer = EEGConformer(n_classes=N_CLS, n_channels=N_CH, T=T_LEN).to(DEVICE)
total_params = sum(p.numel() for p in conformer.parameters())
print(f'✅ EEG Conformer | Parameters: {total_params:,}')

## 🏋️ Training Loop

In [ ]:
def train_model(model, train_loader, test_loader, model_name,
                n_epochs=50, lr=1e-3, device=DEVICE):
    """
    Train loop with:
    - AdamW optimizer + cosine LR schedule
    - Label smoothing (reduces overconfidence)
    - Early stopping (patience=10)
    - Best model checkpoint
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_acc, patience_cnt, patience = 0.0, 0, 10
    train_losses, val_accs = [], []

    for epoch in range(1, n_epochs + 1):
        # ── Train ────────────────────────────────────────────────────────
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss   = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clip
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()

        # ── Validate ─────────────────────────────────────────────────────
        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(device)
                out = model(xb).argmax(dim=1).cpu().numpy()
                preds.extend(out)
                labels.extend(yb.numpy())

        acc = accuracy_score(labels, preds)
        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)
        val_accs.append(acc)

        if epoch % 5 == 0:
            print(f'  Epoch {epoch:3d}/{n_epochs} | Loss: {avg_loss:.4f} | Val Acc: {acc*100:.2f}%')

        # ── Early stopping + checkpoint ───────────────────────────────────
        if acc > best_acc:
            best_acc    = acc
            patience_cnt = 0
            torch.save(model.state_dict(), MODEL_PATH / f'{model_name}_best.pt')
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f'  Early stopping at epoch {epoch}')
                break

    print(f'\n🏆 {model_name} Best Validation Accuracy: {best_acc*100:.2f}%')
    return train_losses, val_accs, best_acc

print('✅ Training function defined.')

In [ ]:
results_dl = {}

# ── Train EEGNet ─────────────────────────────────────────────────────────
print('\n🚀 Training EEGNet...')
eegnet = EEGNet(n_classes=N_CLS, n_channels=N_CH, T=T_LEN).to(DEVICE)
losses1, accs1, best1 = train_model(eegnet, train_loader, test_loader, 'EEGNet')
results_dl['EEGNet'] = best1

# ── Train CNN-LSTM ───────────────────────────────────────────────────────
print('\n🚀 Training CNN-LSTM...')
cnn_lstm = CNNLSTM(n_classes=N_CLS, n_channels=N_CH, T=T_LEN).to(DEVICE)
losses2, accs2, best2 = train_model(cnn_lstm, train_loader, test_loader, 'CNN-LSTM')
results_dl['CNN-LSTM'] = best2

# ── Train EEG Conformer ──────────────────────────────────────────────────
print('\n🚀 Training EEG Conformer...')
conformer = EEGConformer(n_classes=N_CLS, n_channels=N_CH, T=T_LEN).to(DEVICE)
losses3, accs3, best3 = train_model(conformer, train_loader, test_loader, 'Conformer', lr=5e-4)
results_dl['EEG Conformer'] = best3

print('\n📊 Final Results:')
for name, acc in results_dl.items():
    print(f'   {name}: {acc*100:.2f}%')

## 📊 Training Curves + Confusion Matrix

In [ ]:
# ── Training curves ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, losses, accs, color in [
    ('EEGNet',       losses1, accs1, '#2196F3'),
    ('CNN-LSTM',     losses2, accs2, '#4CAF50'),
    ('EEG Conformer',losses3, accs3, '#E91E63'),
]:
    axes[0].plot(losses, label=name, color=color)
    axes[1].plot([a * 100 for a in accs], label=name, color=color)

axes[0].set_title('Training Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].set_title('Validation Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_PATH / 'dl_training_curves.png', dpi=150)
plt.show()

# ── Best model confusion matrix ──────────────────────────────────────────
best_model_name = max(results_dl, key=results_dl.get)
model_map = {'EEGNet': eegnet, 'CNN-LSTM': cnn_lstm, 'EEG Conformer': conformer}
best_model = model_map[best_model_name]

# Load best checkpoint
best_model.load_state_dict(torch.load(MODEL_PATH / f'{best_model_name}_best.pt'))
best_model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        out = best_model(xb.to(DEVICE)).argmax(dim=1).cpu().numpy()
        all_preds.extend(out)
        all_labels.extend(yb.numpy())

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=tasks, yticklabels=tasks, ax=ax)
ax.set_title(f'Confusion Matrix — {best_model_name}', fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(OUTPUT_PATH / f'confusion_matrix_{best_model_name}.png', dpi=150)
plt.show()

print(f'\n🏆 Best deep learning model: {best_model_name} ({results_dl[best_model_name]*100:.2f}%)')
print(classification_report(all_labels, all_preds, target_names=tasks))

---
## ✅ Summary

| Model | Notes |
|---|---|
| **EEGNet** | Compact, good for small datasets, ~2K params |
| **CNN-LSTM** | Good at temporal dynamics, bidirectional |
| **EEG Conformer** | Best at long-range dependencies, most expressive |

Saved checkpoints: `models/<ModelName>_best.pt`

➡️ **Next:** Run `04_generalization.ipynb` for LOSO cross-validation and domain adaptation.